# 05 · 误差分析与未采用的方法

最终模型之后，检查还有没有可挖的信息，并记录试过但没有采用的方法：

1. **残差诊断**：模型在哪些分组上系统性地高估或低估？
2. **其他模型族**：LightGBM 与神经网络 TabM-mini 能否与 CatBoost 互补？
3. **跨实验融合**：把所有保存过的模型放在一起，用嵌套交叉验证挑选融合，能否超过最终方案？
4. **测试集软伪标签**：学习曲线显示模型受数据量限制，用 45,000 个无标签测试行能否补充？
5. **公开榜噪声与运行间差异**：为什么选模只看五折，以及决策阈值从何而来。

| notebook | 内容 |
|---|---|
| **01_data_exploration** | 原始数据读取、审计、探索分析、数据质量检查、固定五折 |
| **02_feature_engineering** | 全部特征的定义、三轮特征消融、保存特征库 |
| **03_model_selection** | 基线模型（逻辑回归、XGBoost、CatBoost、AutoGluon）、线性模型的数值变换、CatBoost 调参与训练方式筛选 |
| **04_final_model** | 最终五折模型、多种子、学习曲线、全量重训与最终提交 |
| **05_error_analysis** | 残差诊断、LightGBM 与神经网络、跨实验融合、伪标签、公开榜噪声与运行间差异 |

原始数据在 `data/`；每本 notebook 的结果保存在 `artifacts/` 下与它同名的目录。

In [ ]:
# 环境、路径与通用函数（5 本 notebook 相同）
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False


def find_project_dir():
    candidates = []
    if os.environ.get('HMDA_PROJECT_DIR'):
        candidates.append(Path(os.environ['HMDA_PROJECT_DIR']))
    candidates.append(Path('/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis'))
    here = Path.cwd().resolve()
    candidates += [here, *here.parents]
    for candidate in candidates:
        if (candidate / 'data' / 'train.csv').is_file() and (candidate / 'data' / 'test.csv').is_file():
            return candidate.expanduser().resolve()
    raise FileNotFoundError('找不到项目目录（应包含 data/train.csv 与 data/test.csv）；请设置环境变量 HMDA_PROJECT_DIR')


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / 'data'                                        # 原始 CSV（只读）
ARTIFACTS = PROJECT_DIR / 'artifacts'                                  # 每本 notebook 的结果写入同名子目录
PREPARED_DIR = ARTIFACTS / '01_data_exploration'                       # 01：固定数据接口
FEATURE_DIR = ARTIFACTS / '02_feature_engineering' / 'feature_store'   # 02：特征库
SEED = 42
CATBOOST_VERSION = '1.2.10'
GPU_NAME = None
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})


def ensure_catboost():
    """只在训练时调用：安装并导入固定版本的 CatBoost。"""
    global catboost
    try:
        installed = importlib.metadata.version('catboost')
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != CATBOOST_VERSION:
        if 'catboost' in sys.modules:
            raise RuntimeError('当前运行时已导入另一版 CatBoost；请重启运行时后从头运行')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'catboost=={CATBOOST_VERSION}'])
        importlib.invalidate_caches()
    import catboost as catboost_module
    if catboost_module.__version__ != CATBOOST_VERSION:
        raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}，实际为 {catboost_module.__version__}')
    catboost = catboost_module
    return catboost_module


def ensure_gpu():
    """只在训练时调用：确认 Colab 分配了 NVIDIA GPU。"""
    global GPU_NAME
    if shutil.which('nvidia-smi') is None:
        raise RuntimeError('训练需要 GPU：Colab「更改运行时类型」选 T4 GPU 后从头运行')
    probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           text=True, capture_output=True, check=False)
    if probe.returncode != 0 or not probe.stdout.strip():
        raise RuntimeError('NVIDIA GPU 不可用：请选择 T4 GPU 后从头运行')
    GPU_NAME = probe.stdout.strip().splitlines()[0]
    return GPU_NAME


print(f'项目：{PROJECT_DIR}')
print(f'Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')

**训练开关**：默认全部为 `False`，只读取 `artifacts/` 中已保存的结果并生成表格和图（Colab 普通 CPU 运行时即可，几分钟跑完）。把某一项改为 `True` 并选择 **T4 GPU** 运行时，会按原协议重新训练对应部分；每个模型训练完立即写入 `artifacts/`，断线后从头运行会跳过已完成的部分。

In [ ]:
TRAIN = {
    'lightgbm': False,       # LightGBM 五折（CPU 即可，约 30 分钟）
    'tabm': False,           # TabM-mini 神经网络（GPU，第 1 折约 5 分钟）
    'pseudo_label': False,   # 软伪标签逐折验证（GPU，约 2 小时；验证通过才做全量部分）
}

## 0. 读取固定数据接口与特征库

In [ ]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


manifest_path = PREPARED_DIR / 'manifest.json'
schema_path = PREPARED_DIR / 'schema.json'
if not manifest_path.is_file() or not schema_path.is_file():
    raise FileNotFoundError(
        f'缺少 {manifest_path} 或 {schema_path}；请先运行 01_data_exploration.ipynb'
    )
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
schema = json.loads(schema_path.read_text(encoding='utf-8'))
require(manifest['schema_version'] == schema['schema_version'] == '1.0', '数据契约版本不匹配')
require(manifest['seed'] == SEED and manifest['n_splits'] == 5, '五折配置不匹配')
require(set(manifest['source_files']) == {'train.csv', 'test.csv', 'sample_submission.csv'},
        '原始文件清单不完整')
expected_prepared = {
    'train_features.csv', 'test_features.csv', 'train_labels.csv',
    'test_ids.csv', 'folds.csv', 'audit.csv', 'schema.json',
}
require(set(manifest['output_files']) == expected_prepared, '准备文件清单不完整')
for name, info in manifest['source_files'].items():
    path = DATA_DIR / name
    require(path.is_file(), f'缺少原始数据：{path}')
    require(sha256_file(path) == info['sha256'], f'原始文件已变化：{name}；请先重跑 01')
for name, info in manifest['output_files'].items():
    path = PREPARED_DIR / name
    require(path.is_file(), f'缺少准备文件：{path}')
    require(sha256_file(path) == info['sha256'], f'准备文件哈希不匹配：{name}；请先重跑 01')

FEATURE_COLUMNS = schema['feature_columns']
NUMERIC_COLUMNS = schema['numeric_columns']
CATEGORICAL_COLUMNS = schema['categorical_columns']
require(len(FEATURE_COLUMNS) == 56 and len(NUMERIC_COLUMNS) == 13
        and len(CATEGORICAL_COLUMNS) == 43, '预期为 56 列：13 数值、43 类别')
require(set(FEATURE_COLUMNS) == set(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS),
        '数值/类别列分组不完整')
require(not {'id', 'accepted'} & set(FEATURE_COLUMNS), 'ID 或标签不能作为特征')


def read_features(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    require(frame.columns.tolist() == FEATURE_COLUMNS, f'特征列或顺序不匹配：{path.name}')
    for column in NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column].replace('', np.nan), errors='raise',
        ).astype('float64')
    require(not np.isinf(frame[NUMERIC_COLUMNS].to_numpy()).any(), '数值列含无穷大')
    require(not frame[CATEGORICAL_COLUMNS].eq('').any().any(), '类别列含空字符串')
    return frame


X_base = read_features(PREPARED_DIR / 'train_features.csv')
X_test_base = read_features(PREPARED_DIR / 'test_features.csv')
labels = pd.read_csv(PREPARED_DIR / 'train_labels.csv', dtype=str, keep_default_na=False)
test_id_frame = pd.read_csv(PREPARED_DIR / 'test_ids.csv', dtype=str, keep_default_na=False)
fold_frame = pd.read_csv(PREPARED_DIR / 'folds.csv', dtype=str, keep_default_na=False)
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype=str, keep_default_na=False)

require(labels.columns.tolist() == ['id', 'accepted'], '训练标签列不匹配')
require(test_id_frame.columns.tolist() == ['id'], '测试 ID 列不匹配')
require(fold_frame.columns.tolist() == ['id', 'fold'], '折号文件列不匹配')
require(sample_submission.columns.tolist() == ['id', 'accepted'], '提交模板列不匹配')
require(len(X_base) == len(labels) == len(fold_frame) == 105000, '训练行数不匹配')
require(len(X_test_base) == len(test_id_frame) == len(sample_submission) == 45000,
        '测试行数不匹配')

train_ids = labels['id']
test_ids = test_id_frame['id']
require(train_ids.is_unique and test_ids.is_unique, 'ID 重复')
require(train_ids.ne('').all() and test_ids.ne('').all(), 'ID 为空')
require(train_ids.equals(fold_frame['id']), '折号 ID 与训练行错位')
require(test_ids.equals(sample_submission['id']), '测试 ID 与提交模板顺序不符')
require(set(train_ids).isdisjoint(test_ids), '训练与测试 ID 重叠')
require(set(labels['accepted']) == {'0', '1'}, 'accepted 需同时包含 0 和 1')
raw_train_identity = pd.read_csv(
    DATA_DIR / 'train.csv', usecols=['id', 'accepted'],
    dtype=str, keep_default_na=False,
)
raw_test_identity = pd.read_csv(
    DATA_DIR / 'test.csv', usecols=['id'], dtype=str, keep_default_na=False,
)
require(train_ids.equals(raw_train_identity['id'])
        and labels['accepted'].equals(raw_train_identity['accepted']),
        '准备标签与原始 train.csv 的 ID/标签不一致')
require(test_ids.equals(raw_test_identity['id']),
        '准备测试 ID 与原始 test.csv 不一致')
y = labels['accepted'].astype('int8')
require(set(fold_frame['fold']) == {'1', '2', '3', '4', '5'}, '折号应为 1..5')
fold_assignment = fold_frame['fold'].astype('int8').to_numpy()

expected_folds = np.zeros(len(X_base), dtype=np.int8)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (_, valid_index) in enumerate(splitter.split(X_base, y), start=1):
    expected_folds[valid_index] = fold
require(np.array_equal(fold_assignment, expected_folds), '折号并非固定 seed=42 五折')
DATA_HASHES = {
    'manifest.json': sha256_file(manifest_path),
    **{name: info['sha256'] for name, info in manifest['output_files'].items()},
}
print(f'数据核对通过：训练 {len(X_base):,}，测试 {len(X_test_base):,}，56 个原始特征。')
display(pd.crosstab(fold_assignment, y, rownames=['fold'], colnames=['accepted']))

def checked_probabilities(model, features):
    require(list(model.classes_) == [0, 1], '类别顺序不是 [0, 1]')
    probability = np.asarray(model.predict_proba(features)[:, 1], dtype=float)
    require(probability.ndim == 1 and len(probability) == len(features)
            and np.isfinite(probability).all()
            and ((probability >= 0) & (probability <= 1)).all(),
            '预测概率无效')
    return probability


def write_json(path, payload):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def fixed_config(output_dir, config):
    """首次运行写入配置；之后配置必须完全相同，避免把不同实验的结果混在一个目录。"""
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / 'config.json'
    if path.is_file():
        saved = json.loads(path.read_text(encoding='utf-8'))
        require(saved == config,
                f'{output_dir.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        write_json(path, config)

In [ ]:
# 读取 02 生成的特征库（只在重新训练时真正需要；只读结果时缺少也不影响）
def load_feature_store():
    meta_path = FEATURE_DIR / 'feature_store.json'
    require(meta_path.is_file(), '缺少特征库：请先运行 02_feature_engineering.ipynb')
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    require(meta['data_hashes'] == DATA_HASHES, '特征库与 01 的固定数据接口不一致：请重跑 02')
    stores = {}
    for part, expected_rows in [('train', len(X_base)), ('test', len(X_test_base))]:
        path = FEATURE_DIR / f'blocks_{part}.parquet'
        require(sha256_file(path) == meta['files'][path.name], f'{path.name} 哈希不符：请重跑 02')
        frame = pd.read_parquet(path)
        require(len(frame) == expected_rows, f'{path.name} 行数不符')
        for column in frame.columns:
            frame[column] = frame[column].astype(object if column in meta['categorical'] else 'float64')
        stores[part] = frame
    blocks = {name: (stores['train'][cols].reset_index(drop=True), stores['test'][cols].reset_index(drop=True))
              for name, cols in meta['blocks'].items()}
    return blocks, set(meta['categorical']), meta


def variant_frames(blocks):
    """原始 56 列 + 指定的特征块；与 02 中的定义相同。"""
    train_parts = [X_base] + [BLOCKS[b][0] for b in blocks]
    test_parts = [X_test_base] + [BLOCKS[b][1] for b in blocks]
    train = pd.concat(train_parts, axis=1)
    test = pd.concat(test_parts, axis=1)
    require(train.columns.is_unique and train.columns.equals(test.columns), '变体列不一致')
    require(train.iloc[:, :len(FEATURE_COLUMNS)].equals(X_base), '原始 56 列被修改')
    cats = CATEGORICAL_COLUMNS + [c for c in train.columns if c in NEW_CATEGORICAL]
    return train, test, cats


if (FEATURE_DIR / 'feature_store.json').is_file():
    BLOCKS, NEW_CATEGORICAL, FEATURE_META = load_feature_store()
    print(f"特征库：{len(BLOCKS)} 个特征块、{sum(len(c) for c in FEATURE_META['blocks'].values())} 列；"
          f"特征集 {list(FEATURE_META['feature_sets'])}")
else:
    BLOCKS, NEW_CATEGORICAL, FEATURE_META = None, set(), None
    print('尚未生成特征库（02）：只读取已保存结果不受影响；需要重新训练时请先运行 02。')

In [ ]:
# 只读的结果工具：读取每个（模型/变体, 折）保存的 JSON，做同折配对比较
def load_runs(directory, pattern='*__fold?.json'):
    directory = Path(directory)
    paths = sorted(directory.glob(pattern))
    require(paths, f'{directory} 中没有结果文件：请确认 artifacts 已从 Drive 同步，或打开对应的训练开关')
    return pd.DataFrame([json.loads(p.read_text(encoding='utf-8')) for p in paths])


def paired_table(runs, key, baseline, min_gain=0.0005, min_folds=4):
    auc = runs.pivot(index='fold', columns=key, values='valid_auc')
    require(baseline in auc.columns, f'缺少基线 {baseline}')
    delta = auc.sub(auc[baseline], axis=0)
    table = pd.DataFrame({
        'folds': auc.notna().sum(),
        'mean_auc': auc.mean(),
        'delta_vs_base': delta.mean(),
        'folds_better': (delta > 0).sum(),
    })
    table['keep'] = ((table['folds'] == 5) & (table['delta_vs_base'] >= min_gain)
                     & (table['folds_better'] >= min_folds))
    table.loc[baseline, 'keep'] = False
    return table.sort_values('mean_auc', ascending=False), delta


def plot_deltas(delta, baseline, title):
    others = delta.drop(columns=[baseline])
    order = others.mean().sort_values().index.tolist()
    means = others[order].mean() * 1e4
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(order) + 1.6))
    ax.barh(range(len(order)), means, color=['#4c72b0' if v > 0 else '#c44e52' for v in means])
    for i, name in enumerate(order):
        ax.scatter(others[name] * 1e4, np.full(len(others), i), color='black', s=12, zorder=3)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_yticks(range(len(order)), order)
    ax.set_xlabel(f'AUC change vs {baseline} (x1e-4); bar = 5-fold mean, dots = single folds')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()



# 读取五折验证预测（OOF）：每个文件是一个外层折的验证行
def read_oof(directory, group):
    frames = [pd.read_csv(Path(directory) / f'{group}__fold{k}__valid.csv').assign(fold=k) for k in range(1, 6)]
    return pd.concat(frames).sort_values('id', kind='stable').reset_index(drop=True)


def fold_auc(frame, probability):
    probability = np.asarray(probability)
    return pd.Series({k: roc_auc_score(frame.loc[frame['fold'].eq(k), 'accepted'],
                                       probability[frame['fold'].eq(k).to_numpy()]) for k in range(1, 6)})


def mean_test(paths):
    frames = [pd.read_csv(p, dtype={'id': str}) for p in paths]
    for f in frames:
        require(f['id'].equals(sample_submission['id']), f'测试 ID 顺序不一致：{Path(paths[0]).name}')
    return np.mean([f['accepted'].to_numpy() for f in frames], axis=0)


CV_DIR = ARTIFACTS / '04_final_model' / 'cv_models'      # 五折模型（M0 与 M1–M3 × 3 个种子）
FULL_DIR = ARTIFACTS / '04_final_model' / 'full_refit'   # 学习曲线与全量模型
CAT_BASE_GROUPS = ['M1_C11_G7', 'M2_C7_G7', 'M3_C11_lean']
SEED_TAGS = {42: '', 2027: '_s2027', 31415: '_s31415'}

## 1. 残差诊断

用 M3 的五折验证预测，按分组计算**标准化残差平方的均值** Σ(y − p)² / Σp(1 − p) 在各组上的平均：≈ 1 表示该分组上模型已经校准、没有遗漏的信息；明显大于随机分组说明模型缺了这部分信息。

In [ ]:
m3 = read_oof(CV_DIR, 'M3_C11_lean')
diag = X_base.assign(id=train_ids.astype('int64').to_numpy(), accepted=y.to_numpy()).merge(
    m3[['id', 'prob']], on='id', validate='one_to_one')
residual = diag['accepted'] - diag['prob']
variance = diag['prob'] * (1 - diag['prob'])


def dispersion(groups, min_rows):
    t = pd.DataFrame({'g': np.asarray(groups).astype(str), 'r': residual.to_numpy(), 'v': variance.to_numpy()})
    t = t.groupby('g').agg(r=('r', 'sum'), v=('v', 'sum'), n=('r', 'size'))
    t = t[t['n'] >= min_rows]
    return float((t['r'] ** 2 / t['v']).mean()), len(t)


ltv = diag['loan_to_value_ratio']
product_flags = ['business_or_commercial_purpose', 'open-end_line_of_credit', 'reverse_mortgage',
                 'interest_only_payment', 'balloon_payment', 'negative_amortization', 'other_nonamortizing_features']
tract_columns = ['county_code', 'tract_population', 'tract_minority_population_percent',
                 'tract_to_msa_income_percentage', 'tract_owner_occupied_units']
rng = np.random.default_rng(SEED)
groupings = [
    ('普查区', diag[tract_columns].astype(str).agg('|'.join, axis=1), 5),
    ('县', diag['county_code'], 5),
    ('MSA', diag['derived_msa-md'], 5),
    ('州', diag['state_code'], 5),
    ('机构画像：AUS × 提交方式 × 支付对象 × 信用评分类型 × 预批',
     diag[['aus-1', 'submission_of_application', 'initially_payable_to_institution',
           'applicant_credit_score_type', 'preapproval']].astype(str).agg('|'.join, axis=1), 5),
    ('随机分组（对照，约 8000 组）', rng.integers(0, 8000, len(diag)), 5),
    ('贷款类型 × DTI', diag['loan_type'] + '|' + diag['debt_to_income_ratio'], 30),
    ('LTV 区间 × DTI', pd.cut(ltv, [0, 60, 80, 90, 95, 97, 100, np.inf]).astype(str) + '|' + diag['debt_to_income_ratio'], 30),
    ('申请人 × 共同申请人年龄', diag['applicant_age'] + '|' + diag['co-applicant_age'], 30),
    ('产品标志组合', diag[product_flags].astype(str).agg('|'.join, axis=1), 30),
    ('随机分组（对照，300 组）', rng.integers(0, 300, len(diag)), 30),
]
display(pd.DataFrame([{'分组': name, '标准化残差平方均值': dispersion(groups, min_rows)[0],
                       '组数': dispersion(groups, min_rows)[1], '最少行数': min_rows}
                      for name, groups, min_rows in groupings]).round(3))

**结论**：地理（普查区、县、MSA）、「机构画像」、贷款类型 × DTI、LTV × DTI 都与随机分组相当，模型在这些方向上不缺信息。只有产品标志组合偏高（仅 16 组，证据弱），它成了第三轮特征（02）的候选之一，消融后同样无效。

### 「异常值」与标签噪声

模型在这些组上的平均预测与实际批准率一致——异常值是信号，清理会丢失信息；模型很有把握却判错的行很少，标签噪声不是问题。

In [ ]:
loan_value = diag['loan_amount'] / diag['property_value']
cases = {
    'LTV 缺失': ltv.isna(), 'LTV > 150': ltv > 150,
    '普查区人口 = 0': diag['tract_population'].eq(0), '地区收入 = 0': diag['ffiec_msa_md_median_family_income'].eq(0),
    '贷款额 / 房价 > 2': loan_value > 2, '收入 ≤ 0': diag['income'] <= 0,
}
rows = []
for name, mask in cases.items():
    rows.append({'情况': name, '行数': int(mask.sum()), '实际批准率': diag.loc[mask, 'accepted'].mean(),
                 '模型平均预测': diag.loc[mask, 'prob'].mean(),
                 '残差 z': residual[mask].sum() / np.sqrt(variance[mask].sum())})
display(pd.DataFrame(rows).round(3))
for threshold in [0.95, 0.98]:
    wrong = ((diag['prob'] > threshold) & diag['accepted'].eq(0)) | ((diag['prob'] < 1 - threshold) & diag['accepted'].eq(1))
    print(f'把握 ≥ {threshold:.0%} 却判错：{int(wrong.sum())} 行（{wrong.mean():.2%}）')

key_columns = ['county_code', 'tract_population', 'tract_minority_population_percent',
               'tract_to_msa_income_percentage', 'tract_owner_occupied_units', 'loan_amount', 'income', 'applicant_age']
both = pd.concat([X_base[key_columns], X_test_base[key_columns]], ignore_index=True).astype(str).agg('|'.join, axis=1)
group_size = both.map(both.value_counts()).iloc[:len(X_base)].to_numpy()
print(f'与其他申请「普查区 + 金额 + 收入 + 年龄」完全相同的训练行：{(group_size >= 2).mean():.2%}')

## 2. 其他模型族

### 2.1 LightGBM

G7 特征（与 M1/M2 相同），两个容量（31 / 127 个叶子），协议与 CatBoost 相同：内层早停 → 外层重训。先看单独的五折 AUC，再与最终 CatBoost（B_seeds）按预设权重融合：权重 0 = 只用 CatBoost，1 = 只用 LightGBM。

In [ ]:
EXPERIMENT_ID = 'lightgbm'
FOLDS = [1, 2, 3, 4, 5]
INNER_STOP_FRACTION = 0.15
MAX_ITERATIONS = 10000
OD_WAIT = 300
OUTPUT_DIR = ARTIFACTS / '05_error_analysis' / EXPERIMENT_ID

NO_FREQ = ['dti', 'ltv', 'income', 'missing']
G7 = NO_FREQ + ['precision', 'thresholds', 'implied', 'term']   # 与最终 CatBoost M1/M2 相同的 72 列
LGBM_BLOCKS = G7
LGBM_BASE = dict(
    objective='binary', learning_rate=0.03, n_estimators=MAX_ITERATIONS,
    feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
    lambda_l2=5.0, cat_l2=10.0, cat_smooth=20.0, min_data_per_group=50,
    max_cat_threshold=64, random_state=SEED, n_jobs=os.cpu_count() or 2, verbose=-1,
)
LGBM_CONFIGS = {                      # 不搜索，只取两个容量；融合时两者等权平均
    'L1_lgbm_leaves31': dict(num_leaves=31, min_child_samples=50),
    'L2_lgbm_leaves127': dict(num_leaves=127, min_child_samples=100),
}

In [ ]:
if TRAIN['lightgbm']:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    try:
        import lightgbm as lgb
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'])
        import lightgbm as lgb
    print('LightGBM', lgb.__version__)

    lgb_train, lgb_test, lgb_cats = variant_frames(LGBM_BLOCKS)
    for column in lgb_cats:               # 训练+测试的类别并集作为固定词表（不涉及标签）
        dtype = pd.CategoricalDtype(sorted(set(lgb_train[column]) | set(lgb_test[column])))
        lgb_train[column] = lgb_train[column].astype(dtype)
        lgb_test[column] = lgb_test[column].astype(dtype)
    lgb_train.columns = [c.replace('-', '_') for c in lgb_train.columns]
    lgb_test.columns = lgb_train.columns


    def run_lgbm(name, config, fold):
        fit_idx = np.flatnonzero(fold_assignment != fold)
        valid_idx = np.flatnonzero(fold_assignment == fold)
        started = time.perf_counter()
        inner_fit, inner_stop = train_test_split(
            fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED, stratify=y.iloc[fit_idx])
        probe = lgb.LGBMClassifier(**LGBM_BASE, **config)
        probe.fit(lgb_train.iloc[inner_fit], y.iloc[inner_fit],
                  eval_set=[(lgb_train.iloc[inner_stop], y.iloc[inner_stop])], eval_metric='binary_logloss',
                  callbacks=[lgb.early_stopping(OD_WAIT, verbose=False)])
        best_iteration = int(probe.best_iteration_)
        rounds = int(round(best_iteration / (1 - INNER_STOP_FRACTION)))
        model = lgb.LGBMClassifier(**dict(LGBM_BASE, n_estimators=rounds), **config)
        model.fit(lgb_train.iloc[fit_idx], y.iloc[fit_idx])
        valid_prob = checked_probabilities(model, lgb_train.iloc[valid_idx])
        test_prob = checked_probabilities(model, lgb_test)
        actual = y.iloc[valid_idx]
        stem = OUTPUT_DIR / f'{name}__fold{fold}'
        pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                      'prob': valid_prob}).to_csv(f'{stem}__valid.csv', index=False, float_format='%.12f')
        pd.DataFrame({'id': test_ids.to_numpy(), 'accepted': test_prob}).to_csv(
            f'{stem}__test.csv', index=False, float_format='%.12f')
        result = {
            'group': name, 'fold': fold,
            'definition': {'model': 'lightgbm', 'config': config, 'blocks': LGBM_BLOCKS},
            'feature_count': lgb_train.shape[1], 'inner_best_iteration': best_iteration,
            'early_stopped': best_iteration < MAX_ITERATIONS, 'refit_rounds': rounds,
            'valid_auc': roc_auc_score(actual, valid_prob),
            'valid_logloss': log_loss(actual, valid_prob, labels=[0, 1]),
            'seconds': time.perf_counter() - started, 'lightgbm': lgb.__version__,
            'finished_at_utc': datetime.now(timezone.utc).isoformat(),
        }
        write_json(Path(f'{stem}.json'), result)
        return result


    for fold in FOLDS:
        for name, config in LGBM_CONFIGS.items():
            path = OUTPUT_DIR / f'{name}__fold{fold}.json'
            if path.is_file():
                saved = json.loads(path.read_text(encoding='utf-8'))
                require(saved['definition']['config'] == config, f'{name} 的定义已改变；请改名')
                print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
                continue
            result = run_lgbm(name, config, fold)
            print(f'{name} fold {fold}: AUC={result["valid_auc"]:.6f}  轮数={result["refit_rounds"]}  '
                  f'{result["seconds"]/60:.1f} 分钟', flush=True)
else:
    print("训练开关关闭（TRAIN['lightgbm'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
LGBM_DIR = ARTIFACTS / '05_error_analysis' / 'lightgbm'
lgbm_runs = load_runs(LGBM_DIR, 'L*_lgbm_*__fold?.json')
display(lgbm_runs.pivot(index='fold', columns='group', values='valid_auc').agg(['mean', 'std']).round(5))

cv_reference = read_oof(CV_DIR, 'M0_C11_all')
b_seeds = np.mean([np.mean([read_oof(CV_DIR, b + t)['prob'] for t in SEED_TAGS.values()], axis=0)
                   for b in CAT_BASE_GROUPS], axis=0)
lgbm_oof = [read_oof(LGBM_DIR, g) for g in sorted(lgbm_runs['group'].unique())]
for frame in lgbm_oof:
    require(frame['id'].equals(cv_reference['id']) and frame['fold'].equals(cv_reference['fold']),
            'LightGBM 的 OOF 与 CatBoost 未对齐')
lgbm = np.mean([frame['prob'] for frame in lgbm_oof], axis=0)
base_folds = fold_auc(cv_reference, b_seeds)
blend_rows = []
for weight in [0.0, 0.1, 0.2, 0.3, 1.0]:                  # 只看预设权重，不做搜索
    folds = fold_auc(cv_reference, (1 - weight) * b_seeds + weight * lgbm)
    blend_rows.append({'LightGBM 权重': weight, '五折 AUC': folds.mean(),
                       '相对 B_seeds': (folds - base_folds).mean(),
                       '更好折数': int((folds - base_folds).gt(0).sum())})
display(pd.DataFrame(blend_rows).round(6))
print(f'B_seeds 与 LightGBM 的 OOF 相关系数：{np.corrcoef(b_seeds, lgbm)[0, 1]:.4f}')

**结论**：LightGBM 单独约 0.911，比 CatBoost 低 0.009；与 CatBoost 的预测高度相关，加 10% 持平、更多权重变差，不采用。

### 2.2 神经网络 TabM-mini

32 个成员共享同一个 3 层 MLP，每个成员只在输入端有自己的 ±1 缩放向量（Gorishniy 等 2024 的简化版，PyTorch 实现）。数值列分位数正态化 + 16 段分段线性编码 + 缺失标记，类别列 8 维嵌入，全部只在外层训练部分上拟合。先跑第 1 折：单独 AUC 不到约 0.915 或融合无提升就停止。

In [ ]:
EXPERIMENT_ID = 'tabm'
FOLDS = [1]                 # 先只跑第 1 折试水；值得继续再改为 [1, 2, 3, 4, 5]
OUTPUT_DIR = ARTIFACTS / '05_error_analysis' / EXPERIMENT_ID
CATBOOST_DIR = ARTIFACTS / '04_final_model' / 'cv_models'   # 04 的五折模型结果

NO_FREQ = ['dti', 'ltv', 'income', 'missing']
G7 = NO_FREQ + ['precision', 'thresholds', 'implied', 'term']   # 与最终 CatBoost 相同的 72 列
NN_CONFIG = dict(
    k=32,                   # TabM 的「集成成员」数：32 个共享大部分权重的 MLP
    width=512, depth=3, dropout=0.1,
    cat_dim=8,              # 每个类别列的嵌入维度
    ple_bins=16,            # 数值列分段线性编码的分段数
    rare_min_count=5,       # 外层训练部分出现少于 5 次的类别并入「未知」
    lr=1e-3, weight_decay=1e-4, batch_size=512,
    max_epochs=60, patience=8,
    inner_stop_fraction=0.15,
)
BLEND_WEIGHTS = [0.1, 0.2, 0.3]

In [ ]:
if TRAIN['tabm']:
    ensure_gpu()
    import torch
    import torch.nn as nn
    from sklearn.preprocessing import QuantileTransformer
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    nn_train, nn_test, nn_cats = variant_frames(G7)
    nn_nums = [c for c in nn_train.columns if c not in nn_cats]
    print(f'特征：{nn_train.shape[1]} 列 = {len(nn_nums)} 数值 + {len(nn_cats)} 类别')


    class FoldPreprocessor:
        """只在外层训练部分上拟合：分位数正态化、分段线性编码 (PLE)、缺失标记、类别词表。"""

        def __init__(self, bins, rare_min_count):
            self.bins = bins
            self.rare_min_count = rare_min_count

        def fit(self, frame):
            num = frame[nn_nums].to_numpy(dtype='float64')
            self.quantile = QuantileTransformer(n_quantiles=1000, output_distribution='normal',
                                                subsample=10**9, random_state=SEED)
            self.quantile.fit(num)                      # sklearn 会忽略 NaN
            self.edges = []
            for j in range(num.shape[1]):
                values = num[:, j][~np.isnan(num[:, j])]
                edges = np.unique(np.quantile(values, np.linspace(0, 1, self.bins + 1)))
                self.edges.append(edges if len(edges) >= 2 else np.array([values.min(), values.min() + 1]))
            self.has_missing = np.isnan(num).any(axis=0)
            self.vocab = []
            for column in nn_cats:
                counts = frame[column].value_counts()
                kept = counts.index[counts >= self.rare_min_count]
                self.vocab.append({value: i + 1 for i, value in enumerate(kept)})   # 0 = 未知/稀有
            self.cardinalities = [len(v) + 1 for v in self.vocab]
            return self

        def transform(self, frame):
            num = frame[nn_nums].to_numpy(dtype='float64')
            missing = np.isnan(num)
            normal = np.nan_to_num(np.clip(self.quantile.transform(num), -5, 5), nan=0.0)
            ple_parts = []
            for j, edges in enumerate(self.edges):
                x = num[:, j:j + 1]
                left, right = edges[:-1][None, :], edges[1:][None, :]
                part = np.clip((x - left) / np.maximum(right - left, 1e-12), 0, 1)
                part = np.nan_to_num(part, nan=0.0)
                ple_parts.append(part)
            dense = np.concatenate([normal] + ple_parts + [missing[:, self.has_missing]], axis=1)
            cats = np.stack([frame[c].map(v).fillna(0).to_numpy(dtype='int64')
                             for c, v in zip(nn_cats, self.vocab)], axis=1)
            return dense.astype('float32'), cats


    class TabMMini(nn.Module):
        """TabM-mini：k 个成员共享同一个 MLP，只在输入端各有一个随机 ±1 的缩放向量；预测取 k 个成员概率的平均。"""

        def __init__(self, d_dense, cardinalities, k, width, depth, dropout, cat_dim):
            super().__init__()
            self.embeddings = nn.ModuleList([nn.Embedding(c, cat_dim) for c in cardinalities])
            d_in = d_dense + cat_dim * len(cardinalities)
            self.adapter = nn.Parameter(torch.randint(0, 2, (k, d_in)).float() * 2 - 1)
            layers = []
            for i in range(depth):
                layers += [nn.Linear(d_in if i == 0 else width, width), nn.ReLU(), nn.Dropout(dropout)]
            self.backbone = nn.Sequential(*layers)
            self.head_weight = nn.Parameter(torch.randn(k, width) / width ** 0.5)
            self.head_bias = nn.Parameter(torch.zeros(k))

        def forward(self, dense, cats):
            parts = [dense] + [emb(cats[:, i]) for i, emb in enumerate(self.embeddings)]
            x = torch.cat(parts, dim=1)                       # (B, D)
            x = x[:, None, :] * self.adapter[None]            # (B, k, D)
            h = self.backbone(x)                              # (B, k, width)
            return (h * self.head_weight).sum(-1) + self.head_bias   # (B, k) logits


    def predict(model, dense, cats, batch=4096):
        model.eval()
        out = []
        with torch.no_grad():
            for start in range(0, len(dense), batch):
                d = torch.from_numpy(dense[start:start + batch]).to(DEVICE)
                c = torch.from_numpy(cats[start:start + batch]).to(DEVICE)
                out.append(torch.sigmoid(model(d, c)).mean(1).cpu().numpy())
        return np.concatenate(out).astype('float64')


    def train_fold(fold):
        cfg = NN_CONFIG
        fit_idx = np.flatnonzero(fold_assignment != fold)
        valid_idx = np.flatnonzero(fold_assignment == fold)
        inner_fit, inner_stop = train_test_split(
            fit_idx, test_size=cfg['inner_stop_fraction'], random_state=SEED, stratify=y.iloc[fit_idx])
        prep = FoldPreprocessor(cfg['ple_bins'], cfg['rare_min_count']).fit(nn_train.iloc[fit_idx])
        data = {name: prep.transform(frame) for name, frame in [
            ('fit', nn_train.iloc[inner_fit]), ('stop', nn_train.iloc[inner_stop]),
            ('valid', nn_train.iloc[valid_idx]), ('test', nn_test)]}
        y_fit = torch.from_numpy(y.iloc[inner_fit].to_numpy(dtype='float32')).to(DEVICE)
        y_stop = y.iloc[inner_stop].to_numpy()

        torch.manual_seed(SEED + fold)
        model = TabMMini(data['fit'][0].shape[1], prep.cardinalities, cfg['k'], cfg['width'],
                         cfg['depth'], cfg['dropout'], cfg['cat_dim']).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
        loss_fn = nn.BCEWithLogitsLoss()
        dense_fit = torch.from_numpy(data['fit'][0]).to(DEVICE)
        cats_fit = torch.from_numpy(data['fit'][1]).to(DEVICE)
        best_auc, best_state, best_epoch, waited = -1.0, None, 0, 0
        started = time.perf_counter()
        for epoch in range(1, cfg['max_epochs'] + 1):
            model.train()
            order = torch.randperm(len(dense_fit), device=DEVICE)
            for start in range(0, len(order), cfg['batch_size']):
                batch = order[start:start + cfg['batch_size']]
                logits = model(dense_fit[batch], cats_fit[batch])
                loss = loss_fn(logits, y_fit[batch][:, None].expand_as(logits))
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            stop_auc = roc_auc_score(y_stop, predict(model, *data['stop']))
            if stop_auc > best_auc:
                best_auc, best_epoch, waited = stop_auc, epoch, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                waited += 1
            print(f'  fold {fold} epoch {epoch:2d}: 内层早停集 AUC={stop_auc:.5f}'
                  f'{"  *" if waited == 0 else ""}', flush=True)
            if waited >= cfg['patience']:
                break
        model.load_state_dict(best_state)
        valid_prob = predict(model, *data['valid'])
        test_prob = predict(model, *data['test'])
        require(np.isfinite(valid_prob).all() and np.isfinite(test_prob).all(), '神经网络预测含非有限值')
        actual = y.iloc[valid_idx]
        stem = OUTPUT_DIR / f'TabMMini__fold{fold}'
        pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                      'prob': valid_prob}).to_csv(f'{stem}__valid.csv', index=False, float_format='%.12f')
        pd.DataFrame({'id': test_ids.to_numpy(), 'accepted': test_prob}).to_csv(
            f'{stem}__test.csv', index=False, float_format='%.12f')
        result = {
            'group': 'TabMMini', 'fold': fold, 'config': cfg, 'blocks': G7,
            'dense_dim': int(data['fit'][0].shape[1]), 'categorical_columns': len(nn_cats),
            'best_epoch': best_epoch, 'inner_stop_auc': best_auc,
            'valid_auc': roc_auc_score(actual, valid_prob),
            'valid_logloss': log_loss(actual, valid_prob, labels=[0, 1]),
            'seconds': time.perf_counter() - started, 'device': DEVICE,
            'torch': torch.__version__, 'finished_at_utc': datetime.now(timezone.utc).isoformat(),
        }
        write_json(Path(f'{stem}.json'), result)
        del model, dense_fit, cats_fit
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()
        return result

    fixed_config(OUTPUT_DIR, {
        'purpose': 'TabM-mini (PyTorch) on G7 features; inner 15% early stop, no outer refit',
        'seed': SEED, 'nn_config': NN_CONFIG, 'blocks': G7, 'data_hashes': DATA_HASHES,
    })
    for fold in FOLDS:
        path = OUTPUT_DIR / f'TabMMini__fold{fold}.json'
        if path.is_file():
            saved = json.loads(path.read_text(encoding='utf-8'))
            print(f'TabMMini fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
            continue
        result = train_fold(fold)
        print(f'TabMMini fold {fold}: AUC={result["valid_auc"]:.6f}  最佳 epoch={result["best_epoch"]}  '
              f'{result["seconds"]/60:.1f} 分钟', flush=True)

    nn_done = sorted(json.loads(p.read_text(encoding='utf-8'))['fold']
                     for p in OUTPUT_DIR.glob('TabMMini__fold?.json'))
    require(nn_done, '神经网络还没有完成的折')

    CAT_BASES = ['M1_C11_G7', 'M2_C7_G7', 'M3_C11_lean']
    SEED_SUFFIXES = ['', '_s2027', '_s31415']
    cat_groups = [b + s for b in CAT_BASES for s in SEED_SUFFIXES]
    missing = [g for g in cat_groups for k in range(1, 6)
               if not (CATBOOST_DIR / f'{g}__fold{k}__valid.csv').is_file()]
    require(not missing, f'缺少 04 的 CatBoost 五折结果，例如 {missing[:3]}；请先完成 04')


    def catboost_fold(fold):
        """04 的 B_seeds：每个基础组先对 3 个种子平均，再三组等权。"""
        per_base = []
        for base in CAT_BASES:
            frames = [pd.read_csv(CATBOOST_DIR / f'{base}{s}__fold{fold}__valid.csv') for s in SEED_SUFFIXES]
            for f in frames[1:]:
                require(f['id'].equals(frames[0]['id']), 'CatBoost OOF 行顺序不一致')
            per_base.append(np.mean([f['prob'].to_numpy() for f in frames], axis=0))
        return frames[0][['id', 'accepted']].assign(cat=np.mean(per_base, axis=0))


    rows = []
    for fold in nn_done:
        nn_valid = pd.read_csv(OUTPUT_DIR / f'TabMMini__fold{fold}__valid.csv')
        merged = catboost_fold(fold).merge(nn_valid[['id', 'prob']], on='id', validate='one_to_one')
        require(len(merged) == len(nn_valid), '神经网络与 CatBoost 的验证行不一致')
        row = {'fold': fold,
               'catboost_auc': roc_auc_score(merged['accepted'], merged['cat']),
               'tabm_auc': roc_auc_score(merged['accepted'], merged['prob']),
               'corr': np.corrcoef(merged['cat'], merged['prob'])[0, 1]}
        for w in BLEND_WEIGHTS:
            row[f'blend_w{int(w * 100)}'] = roc_auc_score(merged['accepted'],
                                                         (1 - w) * merged['cat'] + w * merged['prob'])
        rows.append(row)
    table = pd.DataFrame(rows).set_index('fold')
    for w in BLEND_WEIGHTS:
        table[f'delta_w{int(w * 100)}'] = table[f'blend_w{int(w * 100)}'] - table['catboost_auc']
    display(table.round(6))
    summary = table.mean().to_frame('mean').T
    display(summary.round(6))
    table.to_csv(OUTPUT_DIR / 'blend_by_fold.csv')

    gap = summary['catboost_auc'].iloc[0] - summary['tabm_auc'].iloc[0]
    best_w = max(BLEND_WEIGHTS, key=lambda w: summary[f'delta_w{int(w * 100)}'].iloc[0])
    best_delta = summary[f'delta_w{int(best_w * 100)}'].iloc[0]
    print(f'TabM 单独比 CatBoost 低 {gap:.5f}；与 CatBoost 的相关系数 {summary["corr"].iloc[0]:.4f}；'
          f'最好的融合权重 {best_w}：{best_delta:+.6f}')
    if len(nn_done) < 5:
        verdict = ('值得跑完五折：把 FOLDS 改为 [1, 2, 3, 4, 5] 后从头运行（第 1 折会自动跳过）。'
                   if best_delta >= 0.0003 else
                   '第 1 折上融合提升不足 0.0003，建议到此为止，保持 CatBoost 的 B_seeds。')
        print('判断：' + verdict)
    else:
        better = int((table[f'delta_w{int(best_w * 100)}'] > 0).sum())
        print(f'五折：权重 {best_w} 时 {better}/5 折优于 CatBoost。')
        cat_test = []
        for base in CAT_BASES:
            tests = [pd.read_csv(CATBOOST_DIR / f'{base}{s}__fold{k}__test.csv', dtype={'id': str})
                     for s in SEED_SUFFIXES for k in range(1, 6)]
            for t in tests:
                require(t['id'].equals(sample_submission['id']), 'CatBoost 测试 ID 顺序不一致')
            cat_test.append(np.mean([t['accepted'].to_numpy() for t in tests], axis=0))
        cat_test = np.mean(cat_test, axis=0)
        nn_tests = [pd.read_csv(OUTPUT_DIR / f'TabMMini__fold{k}__test.csv', dtype={'id': str})
                    for k in range(1, 6)]
        for t in nn_tests:
            require(t['id'].equals(sample_submission['id']), 'TabM 测试 ID 顺序不一致')
        nn_test = np.mean([t['accepted'].to_numpy() for t in nn_tests], axis=0)
        for w in BLEND_WEIGHTS:
            path = OUTPUT_DIR / f'submission_Bseeds_TabM_w{int(w * 100)}.csv'
            pd.DataFrame({'id': sample_submission['id'],
                          'accepted': (1 - w) * cat_test + w * nn_test}).to_csv(
                path, index=False, float_format='%.12f')
            print('写出', path.name)
        print('只有当某个权重五折平均 ≥ +0.0003 且 ≥ 4/5 折更好时，才替换 B_seeds。')
else:
    print("训练开关关闭（TRAIN['tabm'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
NN_DIR = ARTIFACTS / '05_error_analysis' / 'tabm'
tabm_runs = load_runs(NN_DIR, 'TabMMini__fold?.json')
display(tabm_runs[['fold', 'valid_auc', 'best_epoch', 'inner_stop_auc']].round(6))
display(pd.read_csv(NN_DIR / 'blend_by_fold.csv').round(6))

**结论**：TabM-mini 第 1 折 0.9088，比 CatBoost 低 0.006；与 CatBoost 的预测相关 0.984，按 10%–30% 融合都不提升，未跑完五折。

## 3. 跨实验融合

候选：最终方案 B_seeds、三组各自的 3 种子平均、M0、LightGBM、CPU 基线（CatBoost R1/R2、XGBoost）。在 4 折上用贪心法（logit 平均、可重复选择）挑选融合，在第 5 折上打分，轮换 5 次——得到的是诚实的估计。

In [ ]:
from scipy.special import logit

reference = read_oof(CV_DIR, 'M0_C11_all')
pool = {}
for base in CAT_BASE_GROUPS:
    pool[f'{base} x3 seeds'] = np.mean([read_oof(CV_DIR, base + t)['prob'] for t in SEED_TAGS.values()], axis=0)
pool['B_seeds'] = np.mean([pool[f'{b} x3 seeds'] for b in CAT_BASE_GROUPS], axis=0)
pool['M0'] = reference['prob'].to_numpy()
pool['LightGBM'] = np.mean([read_oof(LGBM_DIR, g)['prob'] for g in ['L1_lgbm_leaves31', 'L2_lgbm_leaves127']], axis=0)
cpu = pd.read_csv(ARTIFACTS / '03_model_selection' / 'baseline_catboost_xgboost' / 'oof_predictions.csv')
cpu = cpu.sort_values('id', kind='stable').reset_index(drop=True)
require(cpu['id'].equals(reference['id']) and cpu['fold'].equals(reference['fold']), 'CPU 基线 OOF 未对齐')
for column in ['CatBoost_R1', 'CatBoost_R2', 'XGBoost']:
    pool[f'CPU {column}'] = cpu[f'prob_{column}'].to_numpy()
names = list(pool)
labels = reference['accepted'].to_numpy()
folds = reference['fold'].to_numpy()
logits = np.column_stack([logit(np.clip(pool[n], 1e-6, 1 - 1e-6)) for n in names])
display(pd.DataFrame({'五折 AUC': [fold_auc(reference, pool[n]).mean() for n in names],
                      '与 B_seeds 的相关': [np.corrcoef(pool[n], pool['B_seeds'])[0, 1] for n in names]},
                     index=names).sort_values('五折 AUC', ascending=False).round(5))


def mean_fold_auc(score, use_folds):
    return np.mean([roc_auc_score(labels[folds == k], score[folds == k]) for k in use_folds])


def greedy(use_folds, steps=15):
    start = names.index('B_seeds')
    chosen, current = [start], logits[:, start].copy()
    best = mean_fold_auc(current, use_folds)
    for _ in range(steps):
        pick = None
        for j in range(len(names)):
            trial = (current * len(chosen) + logits[:, j]) / (len(chosen) + 1)
            score = mean_fold_auc(trial, use_folds)
            if score > best + 1e-6:
                best, pick = score, j
        if pick is None:
            break
        current = (current * len(chosen) + logits[:, pick]) / (len(chosen) + 1)
        chosen.append(pick)
    return chosen


rows = []
for held_out in range(1, 6):
    chosen = greedy([k for k in range(1, 6) if k != held_out])
    blend = logits[:, chosen].mean(axis=1)
    mask = folds == held_out
    rows.append({'留出的折': held_out,
                 '融合 AUC': roc_auc_score(labels[mask], blend[mask]),
                 'B_seeds AUC': roc_auc_score(labels[mask], pool['B_seeds'][mask]),
                 '选中的模型': ', '.join(f'{names[c]}×{chosen.count(c)}' for c in sorted(set(chosen)))})
nested = pd.DataFrame(rows)
nested['差'] = nested['融合 AUC'] - nested['B_seeds AUC']
display(nested.round(5))
print(f"嵌套交叉验证：平均 {nested['差'].mean():+.5f}，{int((nested['差'] > 0).sum())}/5 折更好")

**结论**：旧模型与 B_seeds 高度相关且更弱，嵌套选择几乎总是只选 B_seeds 本身，融合没有可靠提升。

## 4. 测试集软伪标签

对每个外层折，老师 = 只在该折训练部分上训练的 9 个模型对测试集的平均预测；学生 = M3 + 45,000 个软标签测试行（CatBoost `CrossEntropy`，早停集只用真实行）。与不加伪标签的基线在同一次运行、同一折上比较；第 k 折验证标签始终不参与。

In [ ]:
EXPERIMENT_ID = 'pseudo_label'
FOLDS = [1, 2, 3, 4, 5]
RUN_ONLY = None
INNER_STOP_FRACTION = 0.15
MAX_ITERATIONS = 10000
OD_WAIT = 300
OUTPUT_DIR = ARTIFACTS / '05_error_analysis' / EXPERIMENT_ID

GPU_BASE = dict(
    loss_function='Logloss', nan_mode='Min', random_seed=SEED,
    task_type='GPU', devices='0', thread_count=4,
    allow_writing_files=False, verbose=False,
)
ALL_BLOCKS = ['dti', 'ltv', 'income', 'missing', 'freq']
NO_FREQ = ['dti', 'ltv', 'income', 'missing']
FAST = dict(depth=8, learning_rate=0.05, l2_leaf_reg=3,
            bootstrap_type='Bernoulli', subsample=0.8, early_stop=False, iterations=1600)
C7 = dict(depth=8, learning_rate=0.03, l2_leaf_reg=10, bootstrap_type='Bernoulli', subsample=0.8)
C11 = dict(depth=6, learning_rate=0.03, l2_leaf_reg=10, bootstrap_type='Bernoulli', subsample=0.8)
ROUND2 = ['precision', 'thresholds', 'implied', 'term']
G7 = NO_FREQ + ROUND2                      # 02 特征工程中最好的特征集（72 列）
LEAN = NO_FREQ + ['precision', 'thresholds']  # 第二轮只保留单独有效的两块（68 列）
GROUPS = {
    'M0_C11_all': dict(params=C11, blocks=ALL_BLOCKS),   # 第一轮全部特征（公开榜 0.92073 的配置），同轮对照
    'M1_C11_G7': dict(params=C11, blocks=G7),
    'M2_C7_G7': dict(params=C7, blocks=G7),
    'M3_C11_lean': dict(params=C11, blocks=LEAN),
}
BASELINE = 'M0_C11_all'
CATBOOST_DIR = ARTIFACTS / '04_final_model' / 'cv_models'   # 04 的五折模型结果
CAT_BASES = ['M1_C11_G7', 'M2_C7_G7', 'M3_C11_lean']
SEED_SUFFIXES = {42: '', 2027: '_s2027', 31415: '_s31415'}
FULL_DIR = ARTIFACTS / '04_final_model' / 'full_refit'      # 04 的全量模型
PL_GROUP = 'M3_C11_lean'       # 验证阶段只用五折最好的单组
PL_WEIGHT = 1.0                # 每个伪标签行的样本权重（真实行为 1）
RUN_FINAL_PL = 'auto'          # 'auto'：验证通过（≥ +0.0003 且 ≥ 4/5 折）才做全量；也可设 True / False
if RUN_ONLY is not None:
    unknown = set(RUN_ONLY) - set(GROUPS)
    if unknown:
        raise ValueError(f'RUN_ONLY 中有未知组：{sorted(unknown)}')

In [ ]:
if TRAIN['pseudo_label']:
    ensure_catboost()
    ensure_gpu()
    fixed_config(OUTPUT_DIR, {
        'purpose': 'soft pseudo-labelling of test rows (CrossEntropy), fold-honest validation + full-data students',
        'seed': SEED, 'inner_stop_fraction': INNER_STOP_FRACTION,
        'max_iterations': MAX_ITERATIONS, 'od_wait': OD_WAIT,
        'pl_group': PL_GROUP, 'pl_weight': PL_WEIGHT,
        'gpu_base': {k: v for k, v in GPU_BASE.items() if k != 'thread_count'},
        'data_hashes': DATA_HASHES,
    })


    def model_params(spec, iterations, seed, loss='Logloss'):
        params = {k: v for k, v in spec.items() if k not in {'early_stop', 'iterations'}}
        return dict(GPU_BASE, **params, iterations=iterations, random_seed=seed, loss_function=loss)


    def early_stop_refit(spec, train, cats, idx, seed):
        """与五折模型（04）完全相同的协议：idx 内留 15% 早停 → 在整个 idx 上按 b/0.85 轮重训。"""
        inner_fit, inner_stop = train_test_split(idx, test_size=INNER_STOP_FRACTION,
                                                 random_state=SEED, stratify=y.iloc[idx])
        probe = catboost.CatBoostClassifier(**model_params(spec, MAX_ITERATIONS, seed),
                                            eval_metric='Logloss', od_type='Iter',
                                            od_wait=OD_WAIT, use_best_model=True)
        probe.fit(train.iloc[inner_fit], y.iloc[inner_fit], cat_features=cats,
                  eval_set=(train.iloc[inner_stop], y.iloc[inner_stop]))
        best = int(probe.get_best_iteration()) + 1
        rounds = int(round(best / (1 - INNER_STOP_FRACTION)))
        del probe
        gc.collect()
        model = catboost.CatBoostClassifier(**model_params(spec, rounds, seed), use_best_model=False)
        model.fit(train.iloc[idx], y.iloc[idx], cat_features=cats)
        require(int(model.tree_count_) == rounds, '重训轮数不符')
        return model, best, rounds


    def pseudo_label_refit(spec, train, test, cats, idx, pseudo, seed):
        """真实行（idx）+ 全部测试行（软标签 pseudo）一起训练。早停集只取真实行的 15%，
        重训轮数按有效样本量（真实行 + 权重 × 伪标签行）等比放大。"""
        inner_fit, inner_stop = train_test_split(idx, test_size=INNER_STOP_FRACTION,
                                                 random_state=SEED, stratify=y.iloc[idx])

        def stacked(real_idx):
            frame = pd.concat([train.iloc[real_idx], test], ignore_index=True)
            target = np.concatenate([y.iloc[real_idx].to_numpy(dtype='float64'), pseudo])
            weight = np.concatenate([np.ones(len(real_idx)), np.full(len(test), PL_WEIGHT)])
            return frame, target, weight

        X_fit, y_fit, w_fit = stacked(inner_fit)
        probe = catboost.CatBoostClassifier(**model_params(spec, MAX_ITERATIONS, seed, loss='CrossEntropy'),
                                            eval_metric='CrossEntropy', od_type='Iter',
                                            od_wait=OD_WAIT, use_best_model=True)
        probe.fit(X_fit, y_fit, sample_weight=w_fit, cat_features=cats,
                  eval_set=(train.iloc[inner_stop], y.iloc[inner_stop].astype('float64')))
        best = int(probe.get_best_iteration()) + 1
        effective_inner = len(inner_fit) + PL_WEIGHT * len(test)
        effective_full = len(idx) + PL_WEIGHT * len(test)
        rounds = int(round(best * effective_full / effective_inner))
        del probe, X_fit, y_fit, w_fit
        gc.collect()
        X_all, y_all, w_all = stacked(idx)
        model = catboost.CatBoostClassifier(**model_params(spec, rounds, seed, loss='CrossEntropy'),
                                            use_best_model=False)
        model.fit(X_all, y_all, sample_weight=w_all, cat_features=cats)
        require(int(model.tree_count_) == rounds, '伪标签模型轮数不符')
        return model, best, rounds


    def mean_test(paths):
        frames = [pd.read_csv(p, dtype={'id': str}) for p in paths]
        for f in frames:
            require(f['id'].equals(sample_submission['id']), f'测试 ID 顺序不一致：{paths[0]}')
        return np.mean([f['accepted'].to_numpy() for f in frames], axis=0)


    def teacher_for_fold(fold):
        """第 k 折的老师 = 04 中只在第 k 折外层训练部分上训练的 9 个模型对测试集的平均预测。"""
        return mean_test([CATBOOST_DIR / f'{b}{s}__fold{fold}__test.csv'
                          for b in CAT_BASES for s in SEED_SUFFIXES.values()])


    def teacher_valid(fold):
        frames = [pd.read_csv(CATBOOST_DIR / f'{b}{s}__fold{fold}__valid.csv')
                  for b in CAT_BASES for s in SEED_SUFFIXES.values()]
        for f in frames[1:]:
            require(f['id'].equals(frames[0]['id']), '老师 OOF 行顺序不一致')
        return frames[0][['id']].assign(teacher=np.mean([f['prob'].to_numpy() for f in frames], axis=0))

    pl_spec = GROUPS[PL_GROUP]
    pl_train, pl_test, pl_cats = variant_frames(pl_spec['blocks'])
    for fold in [1, 2, 3, 4, 5]:
        fit_idx = np.flatnonzero(fold_assignment != fold)
        valid_idx = np.flatnonzero(fold_assignment == fold)
        pseudo = teacher_for_fold(fold)
        for name in ['A_base', 'A_pseudo']:
            path = OUTPUT_DIR / f'{name}__fold{fold}.json'
            if path.is_file():
                saved = json.loads(path.read_text(encoding='utf-8'))
                print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
                continue
            started = time.perf_counter()
            if name == 'A_base':
                model, best, rounds = early_stop_refit(pl_spec['params'], pl_train, pl_cats, fit_idx, SEED)
            else:
                model, best, rounds = pseudo_label_refit(pl_spec['params'], pl_train, pl_test, pl_cats,
                                                         fit_idx, pseudo, SEED)
            probability = checked_probabilities(model, pl_train.iloc[valid_idx])
            pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': y.iloc[valid_idx].to_numpy(),
                          'prob': probability}).to_csv(OUTPUT_DIR / f'{name}__fold{fold}__valid.csv',
                                                       index=False, float_format='%.12f')
            result = {'name': name, 'fold': fold, 'group': PL_GROUP, 'pl_weight': PL_WEIGHT,
                      'inner_best_iteration': best, 'refit_rounds': rounds,
                      'valid_auc': roc_auc_score(y.iloc[valid_idx], probability),
                      'seconds': time.perf_counter() - started, 'gpu_name': GPU_NAME,
                      'finished_at_utc': datetime.now(timezone.utc).isoformat()}
            write_json(path, result)
            print(f'{name} fold {fold}: AUC={result["valid_auc"]:.6f}  轮数={rounds}  '
                  f'{result["seconds"]/60:.1f} 分钟', flush=True)
            del model
            gc.collect()

    res = pd.DataFrame([json.loads(p.read_text(encoding='utf-8')) for p in OUTPUT_DIR.glob('A_*__fold?.json')])
    table = res.pivot(index='fold', columns='name', values='valid_auc')
    pl_gain = None
    if {'A_base', 'A_pseudo'} <= set(table.columns):
        table['gain_pseudo'] = table['A_pseudo'] - table['A_base']
        blend_rows = {}
        for fold in table.index:
            if not (OUTPUT_DIR / f'A_pseudo__fold{fold}__valid.csv').is_file():
                continue
            student = pd.read_csv(OUTPUT_DIR / f'A_pseudo__fold{fold}__valid.csv')
            merged = teacher_valid(fold).merge(student, on='id', validate='one_to_one')
            auc_teacher = roc_auc_score(merged['accepted'], merged['teacher'])
            auc_mix = roc_auc_score(merged['accepted'], 0.5 * merged['teacher'] + 0.5 * merged['prob'])
            blend_rows[fold] = {'teacher_9models': auc_teacher, 'mix_gain_vs_teacher': auc_mix - auc_teacher}
        table = table.join(pd.DataFrame(blend_rows).T)
        rounds = res.pivot(index='fold', columns='name', values='refit_rounds')
        table['rounds_ratio'] = rounds['A_pseudo'] / rounds['A_base']
        done = table['gain_pseudo'].dropna()
        pl_gain = (done.mean(), int((done > 0).sum()), len(done))
    display(table.round(6))
    if pl_gain:
        print(f'伪标签（单模型对单模型）：平均 {pl_gain[0]:+.6f}，{pl_gain[1]}/{pl_gain[2]} 折为正。'
              f' mix_gain_vs_teacher 表示学生与 9 模型老师各半后相对老师本身的变化（参考）。')

    complete = pl_gain is not None and pl_gain[2] == 5
    passed = complete and pl_gain[0] >= 0.0003 and pl_gain[1] >= 4
    do_final = passed if RUN_FINAL_PL == 'auto' else bool(RUN_FINAL_PL)
    if not do_final:
        print('跳过全量伪标签训练：' + ('验证未完成。' if not complete else
              f'验证未通过（平均 {pl_gain[0]:+.6f}，{pl_gain[1]}/5 折）。最终提交保持 04 的 full_Bseeds。'))
    else:
        ratio = float(table['rounds_ratio'].mean())
        full_teacher = mean_test([FULL_DIR / f'FULL_{b}_seed{s}__test.csv' for b in CAT_BASES for s in SEED_SUFFIXES])
        all_idx = np.arange(len(y))
        for base in CAT_BASES:
            spec = GROUPS[base]
            train, test, cats = variant_frames(spec['blocks'])
            for seed in SEED_SUFFIXES:
                name = f'FULLPL_{base}_seed{seed}'
                path = OUTPUT_DIR / f'{name}.json'
                if path.is_file():
                    print(f'{name}: 已完成，跳过')
                    continue
                full_info = json.loads((FULL_DIR / f'FULL_{base}_seed{seed}.json').read_text(encoding='utf-8'))
                rounds = int(round(full_info['rounds'] * ratio))
                started = time.perf_counter()
                X_all = pd.concat([train, test], ignore_index=True)
                y_all = np.concatenate([y.to_numpy(dtype='float64'), full_teacher])
                w_all = np.concatenate([np.ones(len(train)), np.full(len(test), PL_WEIGHT)])
                model = catboost.CatBoostClassifier(**model_params(spec['params'], rounds, seed, loss='CrossEntropy'),
                                                    use_best_model=False)
                model.fit(X_all, y_all, sample_weight=w_all, cat_features=cats)
                require(int(model.tree_count_) == rounds, '全量伪标签模型轮数不符')
                probability = checked_probabilities(model, test)
                pd.DataFrame({'id': test_ids.to_numpy(), 'accepted': probability}).to_csv(
                    OUTPUT_DIR / f'{name}__test.csv', index=False, float_format='%.12f')
                write_json(path, {'name': name, 'base': base, 'seed': seed, 'rounds': rounds,
                                  'rounds_ratio': ratio, 'seconds': time.perf_counter() - started,
                                  'gpu_name': GPU_NAME, 'finished_at_utc': datetime.now(timezone.utc).isoformat()})
                print(f'{name}: {rounds} 轮  {(time.perf_counter() - started)/60:.1f} 分钟', flush=True)
                del model, X_all
                gc.collect()

        students = np.mean([mean_test([OUTPUT_DIR / f'FULLPL_{b}_seed{s}__test.csv' for s in SEED_SUFFIXES])
                            for b in CAT_BASES], axis=0)
        full_bseeds = np.mean([mean_test([FULL_DIR / f'FULL_{b}_seed{s}__test.csv' for s in SEED_SUFFIXES])
                               for b in CAT_BASES], axis=0)
        print(f'伪标签全量融合 与 04 的 full_Bseeds 测试预测相关系数：{np.corrcoef(students, full_bseeds)[0, 1]:.5f}')
        for name, probability in [('full_PL', students), ('full_PL_mix', 0.5 * students + 0.5 * full_bseeds)]:
            require(np.isfinite(probability).all() and len(probability) == 45000, f'{name} 概率无效')
            path = OUTPUT_DIR / f'submission_{name}.csv'
            pd.DataFrame({'id': sample_submission['id'], 'accepted': probability}).to_csv(
                path, index=False, float_format='%.12f')
            print('写出', path.name)
        print('建议：验证通过时 submission_full_PL.csv 作主选，04 的 submission_full_Bseeds.csv 作备选。')
else:
    print("训练开关关闭（TRAIN['pseudo_label'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
PL_DIR = ARTIFACTS / '05_error_analysis' / 'pseudo_label'
pl_runs = load_runs(PL_DIR, 'A_*__fold?.json')
pl_table = pl_runs.pivot(index='fold', columns='name', values='valid_auc')
pl_table['gain'] = pl_table['A_pseudo'] - pl_table['A_base']
display(pl_table.round(6))
print(f"软伪标签：平均 {pl_table['gain'].mean():+.5f}，{int((pl_table['gain'] > 0).sum())}/5 折为正")

**结论**：+0.00008（3/5），无效。对比 04 的学习曲线：多 25% **真实标签** +0.0012，多 54% **伪标签** ≈ 0——瓶颈是标签信息本身，而真实标签已经全部用上。

## 5. 公开榜噪声与运行间差异

用五折验证预测模拟公开榜：随机抽 13,500 行（≈ 测试集的 30%）计算 AUC。下面同时比较同一配置在两次不同 Colab 会话中的逐折结果，说明为什么小于约 0.0005 的提升被视为不确定。

In [ ]:
labels = reference['accepted'].to_numpy()
new, old = pool['B_seeds'], pool['M0']
rng = np.random.default_rng(SEED)
subset_auc, paired = [], []
for _ in range(300):
    idx = rng.choice(len(labels), 13500, replace=False)
    auc_new = roc_auc_score(labels[idx], new[idx])
    subset_auc.append(auc_new)
    paired.append(auc_new - roc_auc_score(labels[idx], old[idx]))
true_gap = roc_auc_score(labels, new) - roc_auc_score(labels, old)
print(f'全部 105,000 行上 B_seeds 比 M0 高 {true_gap:+.5f}')
print(f'随机 13,500 行（≈ 公开榜）：AUC 标准差 {np.std(subset_auc):.4f}；配对差标准差 {np.std(paired):.4f}；'
      f'公开榜显示顺序相反的概率 {np.mean(np.array(paired) < 0):.0%}')
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(np.array(paired) * 1e4, bins=30, color='#4c72b0')
ax.axvline(true_gap * 1e4, color='black', ls='--', label='difference on all rows')
ax.axvline(0, color='#c44e52', lw=1)
ax.set_xlabel('AUC(B_seeds) - AUC(M0) on 13.5k random rows (x1e-4)')
ax.set_title('How noisy is a 30% public leaderboard?')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

lc = load_runs(FULL_DIR, 'LC_full__fold?.json').set_index('fold')['valid_auc']
m3_runs = load_runs(CV_DIR, 'M3_C11_lean__fold?.json').set_index('fold')['valid_auc']
display(pd.DataFrame({'学习曲线中的 LC_full': lc, '五折模型中的 M3': m3_runs, '差': lc - m3_runs}).round(6))

## 6. 结论

- 数据干净，没有可清洗的错误；有效的数据处理都是「加信息」，且集中在 LTV 上。
- 模型在各个分组上已经校准，换模型族、换训练方式、跨实验融合、伪标签都没有可靠收益。
- 学习曲线显示模型**受标注数据量限制**：最后一步有效的提升来自用上全部训练数据（04）。
- 公开榜噪声约 0.001，选模必须依据五折与学习曲线。

不使用公开的 HMDA 原始数据：测试集本身来自该数据，用它训练会带入测试集的真实标签。